In [31]:
import sys  
sys.path.insert(1, '/home/spuchin/GitHub/the-hand-of-midas')

from clickhouse_connect.driver.asyncclient import AsyncClient as AsyncClickHouseClient
from pandas import DataFrame, read_csv
import plotly.graph_objects as go
from talib._ta_lib import EMA

In [32]:
ohlc: DataFrame = read_csv("/home/spuchin/GitHub/the-hand-of-midas/notebooks/DS-3/macro-candles.csv")
ohlc.head(1)

,open,high,low,close,datetime,open-macro,high-macro,low-macro,close-macro
0,4261.48,4349.99,4261.32,4349.99,2017-08-17 04:00:00,NaN,NaN,NaN,NaN


In [33]:
def is_above(first: float, second: float) -> int:
    return 1 if first > second else 0


EXPONENTIAL_MOVING_AVERAGE: int = 70

ohlc["high-global-trend"] = EMA(ohlc["high"].values, EXPONENTIAL_MOVING_AVERAGE)
ohlc["low-global-trend"] = EMA(ohlc["low"].values, EXPONENTIAL_MOVING_AVERAGE)

candle_columns = ["open", "high", "low", "close", "open-macro", "high-macro", "low-macro", "close-macro"]
for candle_column in candle_columns:
    ohlc[f"is-{candle_column}-above-high-global-trend"] = ohlc.apply(
        lambda row: is_above(
            first=row[candle_column], 
            second=row["high-global-trend"]
        ), 
        axis=1
    )
    ohlc[f"is-{candle_column}-above-low-global-trend"] = ohlc.apply(
        lambda row: is_above(
            first=row[candle_column], 
            second=row["low-global-trend"]
        ), 
        axis=1
    )
    ohlc[f"{candle_column}-to-high-global-trend-ratio"] = (ohlc[candle_column] / ohlc["high-global-trend"] - 1).shift(1)
    ohlc[f"{candle_column}-to-low-global-trend-ratio"] = (ohlc[candle_column] / ohlc["low-global-trend"] - 1).shift(1)

    ohlc[f"is-{candle_column}-above-high-global-trend"] = ohlc[f"is-{candle_column}-above-high-global-trend"].shift(1)
    ohlc[f"is-{candle_column}-above-low-global-trend"] = ohlc[f"is-{candle_column}-above-low-global-trend"].shift(1)

ohlc["high-global-trend"] = ohlc["high-global-trend"].shift(1)
ohlc["low-global-trend"] = ohlc["low-global-trend"].shift(1)

ohlc.tail(1)

,open,high,low,close,datetime,open-macro,high-macro,low-macro,close-macro,high-global-trend,...,high-macro-to-high-global-trend-ratio,high-macro-to-low-global-trend-ratio,is-low-macro-above-high-global-trend,is-low-macro-above-low-global-trend,low-macro-to-high-global-trend-ratio,low-macro-to-low-global-trend-ratio,is-close-macro-above-high-global-trend,is-close-macro-above-low-global-trend,close-macro-to-high-global-trend-ratio,close-macro-to-low-global-trend-ratio
17147,107424.96,107684.13,107309.15,107684.13,2025-06-16 16:00:00,105938.907808,106452.505838,105468.006864,106073.988415,106703.006565,...,-0.003596,0.006144,0.0,0.0,-0.012537,-0.002884,0.0,1.0,-0.007161,0.002544


In [35]:
first_number_of_rows: int = 1200

candlesticks = go.Candlestick(
    x=ohlc["datetime"].head(first_number_of_rows),
    open=ohlc["open-macro"].head(first_number_of_rows),
    high=ohlc["high-macro"].head(first_number_of_rows),
    low=ohlc["low-macro"].head(first_number_of_rows),
    close=ohlc["close-macro"].head(first_number_of_rows),
    showlegend=False
)

global_trend_lines = []
global_trend_columns = ["high-global-trend", "low-global-trend"]
for global_trend_column in global_trend_columns:
    global_trend = go.Scatter(
        x=ohlc["datetime"].head(first_number_of_rows),
        y=ohlc[global_trend_column].head(first_number_of_rows),
        name=global_trend_column
    )
    global_trend_lines.append(global_trend)

global_trend_lines.append(candlesticks)
figure = go.Figure(data=global_trend_lines)

figure.update_layout(xaxis_rangeslider_visible=False)
figure.show()

In [36]:
ohlc.drop(["high-global-trend", "low-global-trend"], axis=1, inplace=True)
ohlc.to_csv("global-trend-lines.csv", index=False)